# Destilar una voz Kokoro a Piper (todo automático)Genera un dataset sintético con una voz Kokoro (Dora/Alex/Santa, español) y entrena una voz **Piper** con ese timbre: calidad Kokoro, velocidad y liviandad Piper, sin grabar ni transcribir nada.**Antes de ejecutar**: Entorno de ejecución → Cambiar tipo de entorno → **T4 GPU**.Tiempo total: ~30 min de generación + 2-4 h de entrenamiento (podés cortar antes y exportar).

In [ ]:
#@title 1. Verificar GPUimport torchassert torch.cuda.is_available(), "Activá la GPU: Entorno de ejecución -> Cambiar tipo -> T4 GPU"print("GPU OK:", torch.cuda.get_device_name(0))

In [ ]:
#@title 2. Elegir la voz Kokoro y cuántos minutos generarVOZ = "ef_dora"  #@param ["ef_dora", "em_alex", "em_santa"]MINUTOS = 35  #@param {type:"slider", min:15, max:90, step:5}print(f"Se destilará {VOZ} con ~{MINUTOS} minutos de audio sintético")

In [ ]:
#@title 3. Instalar Kokoro y descargar su modelo!pip install -q kokoro-onnx==0.5.0 soundfile!wget -q -nc -O /content/kokoro-v1.0.onnx "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx"!wget -q -nc -O /content/voices-v1.0.bin "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin"from kokoro_onnx import Kokorokokoro = Kokoro("/content/kokoro-v1.0.onnx", "/content/voices-v1.0.bin")print("Kokoro cargado")

In [ ]:
#@title 4. Texto fuente: Don Quijote (dominio público) — o subí tu .txtimport os, urllib.requestTEXTO = "/content/texto.txt"if not os.path.exists(TEXTO):    try:        urllib.request.urlretrieve("https://www.gutenberg.org/cache/epub/2000/pg2000.txt", TEXTO)        print("Descargado: Don Quijote (Gutenberg)")    except Exception as e:        print("No se pudo descargar; subí un .txt largo:", e)        from google.colab import files        up = files.upload()        os.rename(list(up.keys())[0], TEXTO)print(os.path.getsize(TEXTO)//1000, "KB de texto")

In [ ]:
#@title 5. Generar el dataset (frases sintéticas + transcripción perfecta)import re, wave, numpy as np, osdef frases(texto, min_c=30, max_c=200):    texto = re.sub(r"\s+", " ", texto)    out = []    for s in re.split(r"(?<=[.!?…])\s+", texto):        s = s.strip()        if min_c <= len(s) <= max_c and not re.search(r"[|_#@{}<>\[\]*]", s):            out.append(s)    return outtexto = open(TEXTO, encoding="utf-8", errors="replace").read()# saltear el prólogo de Gutenberginicio = texto.find("En un lugar de la Mancha")if inicio > 0: texto = texto[inicio:]candidatas = frases(texto)print(len(candidatas), "frases candidatas")os.makedirs("/content/dataset/wavs", exist_ok=True)total, rows = 0.0, []for i, frase in enumerate(candidatas):    if total >= MINUTOS * 60: break    try:        samples, rate = kokoro.create(frase, voice=VOZ, speed=1.0, lang="es")    except Exception:        continue    # remuestrear 24000 -> 22050 (lo que espera Piper)    idx = np.linspace(0, len(samples)-1, int(len(samples)*22050/rate))    data = np.clip(np.interp(idx, np.arange(len(samples)), samples)*32767, -32768, 32767).astype(np.int16)    seg = len(data)/22050    if not 1.0 <= seg <= 20.0: continue    name = f"frase{i:05d}"    with wave.open(f"/content/dataset/wavs/{name}.wav", "wb") as w:        w.setnchannels(1); w.setsampwidth(2); w.setframerate(22050)        w.writeframes(data.tobytes())    rows.append(f"{name}|{frase}")    total += seg    if len(rows) % 50 == 0: print(f"  {len(rows)} frases, {total/60:.1f} min")open("/content/dataset/metadata.csv", "w", encoding="utf-8").write("\n".join(rows)+"\n")print(f"DATASET LISTO: {len(rows)} frases, {total/60:.1f} minutos")

In [ ]:
#@title 6. Instalar Piper (entrenamiento)%cd /content!git clone -q https://github.com/rhasspy/piper.git%cd /content/piper/src/python!pip install -q -e .!pip install -q "pytorch-lightning~=1.9" "espeak-phonemizer" "librosa" "numpy<2"!apt-get install -yq espeak-ng > /dev/null!bash build_monotonic_align.shprint("Piper instalado")

In [ ]:
#@title 7. Preprocesar + checkpoint base español%cd /content/piper/src/python!python -m piper_train.preprocess \  --language es --input-dir /content/dataset --output-dir /content/train_out \  --dataset-format ljspeech --single-speaker --sample-rate 22050!wget -q -nc -O /content/base_es.ckpt "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/es/es_ES/davefx/medium/epoch%3D2218-step%3D562840.ckpt"print("Listo para entrenar")

In [ ]:
#@title 8. Entrenar (2-4 h; podés interrumpir y pasar a la celda 9)%cd /content/piper/src/python!python -m piper_train \  --dataset-dir /content/train_out --accelerator gpu --devices 1 \  --batch-size 16 --validation-split 0.0 --num-test-examples 0 \  --max_epochs 2600 --resume_from_checkpoint /content/base_es.ckpt \  --checkpoint-epochs 5 --precision 32 --quality medium

In [ ]:
#@title 9. Exportar y descargar tu voz Piper con timbre Kokoroimport glob, shutil, osckpts = sorted(glob.glob("/content/train_out/lightning_logs/*/checkpoints/*.ckpt"), key=os.path.getmtime)assert ckpts, "No hay checkpoints: corré la celda 8 al menos unos epochs"%cd /content/piper/src/pythonnombre = f"{VOZ}_piper"!python -m piper_train.export_onnx "{ckpts[-1]}" /content/{nombre}.onnxshutil.copy("/content/train_out/config.json", f"/content/{nombre}.onnx.json")from google.colab import filesfiles.download(f"/content/{nombre}.onnx")files.download(f"/content/{nombre}.onnx.json")print("Copiá ambos archivos a tu carpeta de voces de LoudVox y elegila en Configuración")